# Week 94, Day 6: Review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w094d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('minirepo.py', 'w', encoding='utf-8') as f:
    f.write('REPO = {\n"README.md": \'\'\'# pathkit\n\nSmall path-finding helpers: a graph, shortest routes, CSV loading.\n\n    python -m pathkit.cli map.csv a c\n\'\'\',\n"pathkit/__init__.py": \'\'\'"""A small library of path-finding helpers."""\nfrom pathkit.graph import Graph\nfrom pathkit.search import shortest\n\n__all__ = ["Graph", "shortest"]\n\'\'\',\n"pathkit/graph.py": \'\'\'class Graph:\n    """An undirected graph with a cost on each edge."""\n\n    def __init__(self):\n        self.edges = {}\n\n    def add(self, a, b, cost):\n        self.edges.setdefault(a, {})[b] = cost\n        self.edges.setdefault(b, {})[a] = cost\n\n    def neighbours(self, node):\n        return sorted(self.edges.get(node, {}).items())\n\n\ndef from_pairs(pairs):\n    graph = Graph()\n    for a, b, cost in pairs:\n        graph.add(a, b, cost)\n    return graph\n\'\'\',\n"pathkit/search.py": \'\'\'import heapq\n\nfrom pathkit.graph import Graph\n\n\ndef shortest(graph, start, end):\n    heap, seen = [(0, start, [start])], set()\n    while heap:\n        cost, node, path = heapq.heappop(heap)\n        if node == end:\n            return cost, path\n        if node in seen:\n            continue\n        seen.add(node)\n        for nxt, step in graph.neighbours(node):\n            heapq.heappush(heap, (cost + step, nxt, path + [nxt]))\n    return None\n\n\ndef reachable(graph, start):\n    found, todo = {start}, [start]\n    while todo:\n        for nxt, _ in graph.neighbours(todo.pop()):\n            if nxt not in found:\n                found.add(nxt)\n                todo.append(nxt)\n    return found\n\'\'\',\n"pathkit/io.py": \'\'\'import csv\n\nfrom pathkit.graph import from_pairs\n\n\ndef load_csv(path):\n    with open(path, newline="") as f:\n        rows = [(a, b, float(cost)) for a, b, cost in csv.reader(f)]\n    return from_pairs(rows)\n\'\'\',\n"pathkit/cli.py": \'\'\'import sys\n\nfrom pathkit.io import load_csv\nfrom pathkit.search import shortest\n\n\ndef main(argv=None):\n    argv = sys.argv[1:] if argv is None else argv\n    graph = load_csv(argv[0])\n    result = shortest(graph, argv[1], argv[2])\n    if result is None:\n        print("no route")\n        return 1\n    cost, path = result\n    print(" to ".join(path), cost)\n    return 0\n\'\'\',\n"tests/test_graph.py": \'\'\'from pathkit.graph import Graph, from_pairs\n\n\ndef test_add_is_symmetric():\n    g = Graph()\n    g.add("a", "b", 2)\n    assert g.neighbours("b") == [("a", 2)]\n\n\ndef test_from_pairs():\n    g = from_pairs([("a", "b", 1), ("b", "c", 2)])\n    assert g.neighbours("b") == [("a", 1), ("c", 2)]\n\'\'\',\n"tests/test_search.py": \'\'\'from pathkit.graph import from_pairs\nfrom pathkit.search import shortest\n\n\ndef test_shortest():\n    g = from_pairs([("a", "b", 1), ("b", "c", 2), ("a", "c", 5)])\n    assert shortest(g, "a", "c") == (3, ["a", "b", "c"])\n\'\'\',\n}\n')

**Predict before you run:** what will this print?



In [ ]:
import ast
tree = ast.parse("class Graph:\n    def add(self):\n        pass\n\ndef from_pairs():\n    pass\n")
print([n.name for n in tree.body], [n.name for n in tree.body[0].body])

**Predict before you run:** what will this print?



In [ ]:
from collections import Counter
print(Counter(["ana", "ben", "ana", "chi"]).most_common(1))

**Predict before you run:** what will this print?



In [ ]:
print(sorted(["3.9", "3.13", "3.14"], key=lambda v: tuple(map(int, v.split("."))))[-1])

**Predict before you run:** what will this print?



In [ ]:
import re
print(re.findall(r"[A-Za-z_][\w.]*", "shortest() ignores Graph.add costs"))

**Predict before you run:** what will this print?



In [ ]:
labels = {"type-bug", "easy"}
print("Library" if "docs" not in labels else "Documentation", bool(labels & {"easy", "good first issue"}))

### Exercise: Which tests does a change affect?

Write `affected_tests(graph, module)`: `graph` maps each module to the modules it imports. Return, sorted, the test modules (names starting with `tests.` or whose last part starts with `test_`) that import `module` directly or through other modules.

In [ ]:
def affected_tests(graph, module):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GRAPH = {\"pathkit\": [\"pathkit.graph\", \"pathkit.search\"], \"pathkit.cli\": [\"pathkit.io\", \"pathkit.search\"], \"pathkit.graph\": [],\n         \"pathkit.io\": [\"pathkit.graph\"], \"pathkit.search\": [\"pathkit.graph\"], \"tests.test_graph\": [\"pathkit.graph\"],\n         \"tests.test_search\": [\"pathkit.graph\", \"pathkit.search\"], \"tests.test_cli\": [\"pathkit.cli\"]}\n\ndef test_affected():\n    \"directly or through other modules\"\n    assert affected_tests(GRAPH, \"pathkit.graph\") == [\"tests.test_cli\", \"tests.test_graph\", \"tests.test_search\"]\n    assert affected_tests(GRAPH, \"pathkit.io\") == [\"tests.test_cli\"]\n    assert affected_tests(GRAPH, \"pathkit\") == []")

### Exercise: Reading a NEWS file name

Write `parse_news_path(path)` for paths like `Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-140123.AbCdEf.rst`: return a dict with `section`, `when` (a `datetime`), `issue` (an int) and `nonce`. Raise `ValueError("not a NEWS entry: PATH")` if it doesn't match.

In [ ]:
import re
from datetime import datetime

def parse_news_path(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import datetime\n\ndef test_parse():\n    \"blurb's names, read back\"\n    assert parse_news_path(\"Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-140123.AbCdEf.rst\") == {\n        \"section\": \"Library\", \"when\": datetime(2026, 11, 20, 14, 3, 12), \"issue\": 140123, \"nonce\": \"AbCdEf\"}\n    for bad in [\"Misc/NEWS.d/next/Library/notes.rst\", \"Lib/textwrap.py\"]:\n        try:\n            parse_news_path(bad)\n        except ValueError as e:\n            assert str(e) == f\"not a NEWS entry: {bad}\"\n        else:\n            raise AssertionError(bad)")

### Exercise: A contribution plan

Write `contribution_plan(repo, issue, branches)`. `repo` maps paths to sources (like `REPO`); `issue` has `number`, `title` and `labels`; `branches` maps versions to statuses. Return a list of lines:

1. `"names: ..."`: the words in the title that are defined in the repository, in title order without repeats. A word is a match for `[A-Za-z_][A-Za-z0-9_.]*`; it's defined if it, or its last dotted part, is a key of the definitions index (functions, classes and `Class.method`, as on Day 1). Use the key that matched;
2. `"defined in: ..."`: every `PATH:LINE` for those names, in order, without repeats;
3. `"called by: ..."`: the callers (Day 1's rule, by the name's last dotted part) of each name, in order, without repeats;
4. `"tests to run: ..."`: the test modules affected by the modules those names are defined in, sorted, without repeats;
5. `"kind: KIND"`: `docs` if the labels include `docs`, `feature` if `type-feature`, `security` if `type-security`, otherwise `bug`;
6. `"news section: SECTION"`: `Documentation` for docs, `Security` for security, `Library` otherwise;
7. `"backports: ..."`: Day 3's labels for the kind.

Write `none` for any empty list.

In [ ]:
import ast
import re

def contribution_plan(repo, issue, branches):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from minirepo import REPO\n\nBRANCHES = {\"3.15\": \"feature\", \"3.14\": \"bugfix\", \"3.13\": \"bugfix\", \"3.12\": \"security\"}\n\ndef test_bug():\n    \"a bug in shortest\"\n    issue = {\"number\": 77, \"title\": \"shortest() ignores a cheaper edge after Graph.add\", \"labels\": [\"type-bug\"]}\n    assert contribution_plan(REPO, issue, BRANCHES) == [\n        \"names: shortest, Graph.add\",\n        \"defined in: pathkit/search.py:6, pathkit/graph.py:7\",\n        \"called by: pathkit/cli.py:main, tests/test_search.py:test_shortest, pathkit/graph.py:from_pairs, pathkit/search.py:shortest, pathkit/search.py:reachable, tests/test_graph.py:test_add_is_symmetric\",\n        \"tests to run: tests.test_graph, tests.test_search\",\n        \"kind: bug\",\n        \"news section: Library\",\n        \"backports: needs backport to 3.14, needs backport to 3.13\",\n    ]\n\ndef test_feature():\n    \"a feature with nothing matched\"\n    issue = {\"number\": 80, \"title\": \"Support weighted heuristics\", \"labels\": [\"type-feature\"]}\n    assert contribution_plan(REPO, issue, BRANCHES) == [\"names: none\", \"defined in: none\", \"called by: none\", \"tests to run: none\",\n                                                        \"kind: feature\", \"news section: Library\", \"backports: none\"]\n\ndef test_docs():\n    \"documentation for reachable\"\n    issue = {\"number\": 81, \"title\": \"Document what reachable returns for reachable nodes\", \"labels\": [\"docs\"]}\n    lines = contribution_plan(REPO, issue, BRANCHES)\n    assert lines[:4] == [\"names: reachable\", \"defined in: pathkit/search.py:20\", \"called by: none\", \"tests to run: tests.test_search\"]\n    assert lines[4:6] == [\"kind: docs\", \"news section: Documentation\"]")

> **Before writing any code.** The plan for issue 77 shows something worth noticing: `Graph.add` is called from `from_pairs` and the tests, but the tool also lists `shortest` and `reachable`, which call `set.add`, not `Graph.add`. A real plan reads each caller before trusting it. What would you check first, and which test would you write before touching `shortest`?

### Exercise: Bonus: from issue to merge

Write `timeline(events)`: each event is `(day, kind)` with `day` a `"YYYY-MM-DD"` string and `kind` one of `"issue opened"`, `"comment"`, `"pr opened"`, `"review"`, `"changes pushed"`, `"ci failed"` and `"merged"`, in time order. Return lines: `"days to first response: N"` (from the issue to the first comment or review after it, or `-`), `"days from pull request to merge: N"` (or `-` if either is missing), `"review rounds: N"` (the number of reviews), `"pushes after review: N"`, `"ci failures: N"`, and `"total days: N"` (first event to last).

In [ ]:
from datetime import date

def timeline(events):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("EVENTS = [(\"2026-11-02\", \"issue opened\"), (\"2026-11-04\", \"comment\"), (\"2026-11-09\", \"pr opened\"), (\"2026-11-09\", \"ci failed\"),\n          (\"2026-11-10\", \"changes pushed\"), (\"2026-11-15\", \"review\"), (\"2026-11-16\", \"changes pushed\"), (\"2026-11-22\", \"review\"),\n          (\"2026-11-23\", \"changes pushed\"), (\"2026-11-24\", \"review\"), (\"2026-11-25\", \"merged\")]\n\ndef test_timeline():\n    \"a first contribution\"\n    assert timeline(EVENTS) == [\"days to first response: 2\", \"days from pull request to merge: 16\", \"review rounds: 3\",\n                                \"pushes after review: 2\", \"ci failures: 1\", \"total days: 23\"]\n\ndef test_waiting():\n    \"still waiting\"\n    assert timeline([(\"2026-11-02\", \"issue opened\"), (\"2026-11-09\", \"pr opened\")]) == [\n        \"days to first response: -\", \"days from pull request to merge: -\", \"review rounds: 0\", \"pushes after review: 0\",\n        \"ci failures: 0\", \"total days: 7\"]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## From issue to plan

A first contribution goes better with a plan written before any code: where the code is (search and definitions), what else uses it (callers, so you know what might break), which tests cover it (the test modules that import it, directly or not), what kind of change it is (bug, docs, feature), and so where the NEWS entry goes and which branches it belongs on. Each of those is a tool from this week. Put them together and you have the first comment of a good pull request description.

## Bonus: a contribution's timeline

## Weekly quiz

## Week 94 review

Tick off each one honestly. If one feels shaky, go back to that day.

- I can search a codebase, index its definitions, map its imports, find a reading order and find callers. ([Day 1](w094d1.html))
- I can explain governance, sort licences, read CONTRIBUTING, triage issues and judge a project's health. ([Day 2](w094d2.html))
- I can find my way around CPython's repository and follow its workflow, NEWS entries and backports. ([Day 3](w094d3.html))
- I can build CPython, follow source to bytecode, and compare test runs with a baseline. ([Day 4](w094d4.html))
- I can search for, score and choose a first issue, and comment usefully. ([Day 5](w094d5.html))
- I can turn an issue into a contribution plan, and read a contribution's timeline. (Today)

## Your turn

1. **Write your plan** for the issue you chose yesterday, with your own `contribution_plan` or by hand: names, places, callers, tests, kind, NEWS section, backports.
2. **Reproduce it** on your build, and save the exact commands and output: next week starts there.
3. **Read two merged pull requests** for similar issues, to see what reviewers asked for.

Tomorrow is a **rest day**. Next week is **Open source II**: reproducing the issue properly, writing the fix and its test, opening the pull request, and responding to review.